# Lab 03 - Naive Bayes

Notebook được chạy từ thư mục gốc `03. Naive_Bayes`.

Trên Colab, hãy mount Drive rồi dùng `%cd` để đi đến thư mục này trước khi chọn **Run all**. Hai tệp dữ liệu phải nằm tại `data/tin_nhan.csv` và `data/sinh_vien.csv`.


## Bước 1 - Tính Bayes bằng tay


In [1]:
"""Bước 1: tính xác suất hậu nghiệm bằng công thức Bayes."""


def bayes(prior, likelihood, reverse_likelihood):
    """Trả về P(A|B) từ P(A), P(B|A) và P(B|không A)."""
    numerator = likelihood * prior
    denominator = numerator + reverse_likelihood * (1 - prior)
    return numerator / denominator


print("=== Ví dụ xét nghiệm bệnh ===")
p_benh = 0.01
p_duong_tinh_khi_benh = 0.99
p_duong_tinh_khi_khong_benh = 0.01
print(f"P(Bệnh | Dương tính) = {bayes(p_benh, p_duong_tinh_khi_benh, p_duong_tinh_khi_khong_benh):.4f}")

print("\n=== Ảnh hưởng của xác suất tiên nghiệm ===")
for prior in [0.001, 0.01, 0.05, 0.10, 0.50]:
    posterior = bayes(prior, 0.99, 0.01)
    print(f"P(Bệnh) = {prior:>5.3f} -> P(Bệnh | Dương tính) = {posterior:.4f}")

print("\n=== Ví dụ dây chuyền sản xuất ===")
# A làm 80% sản phẩm và có 1% lỗi; B làm 20% sản phẩm và có 5% lỗi.
p_b_khi_loi = bayes(prior=0.20, likelihood=0.05, reverse_likelihood=0.01)
print(f"P(Dây chuyền B | Sản phẩm lỗi) = {p_b_khi_loi:.4f}")

loi_a = 800 * 0.01
loi_b = 200 * 0.05
print(f"Kiểm tra trên 1000 sản phẩm: B có {loi_b:.0f}/{loi_a + loi_b:.0f} sản phẩm lỗi")


=== Ví dụ xét nghiệm bệnh ===
P(Bệnh | Dương tính) = 0.5000

=== Ảnh hưởng của xác suất tiên nghiệm ===
P(Bệnh) = 0.001 -> P(Bệnh | Dương tính) = 0.0902
P(Bệnh) = 0.010 -> P(Bệnh | Dương tính) = 0.5000
P(Bệnh) = 0.050 -> P(Bệnh | Dương tính) = 0.8390
P(Bệnh) = 0.100 -> P(Bệnh | Dương tính) = 0.9167
P(Bệnh) = 0.500 -> P(Bệnh | Dương tính) = 0.9900

=== Ví dụ dây chuyền sản xuất ===
P(Dây chuyền B | Sản phẩm lỗi) = 0.5556
Kiểm tra trên 1000 sản phẩm: B có 10/18 sản phẩm lỗi


## Bước 2 - Mô hình túi từ


In [2]:
"""Bước 2: biến văn bản thành mô hình túi từ."""

from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "tin_nhan.csv")

print("=== Thông tin dữ liệu ===")
print(f"Kích thước: {df.shape}")
print("Số lượng từng lớp:")
print(df["la_rac"].value_counts().sort_index().to_string())
print(f"Tỷ lệ thư rác: {df['la_rac'].mean():.4f}")

vectorizer = CountVectorizer(token_pattern=r"(?u)\S+")
X = vectorizer.fit_transform(df["noi_dung"])
words = vectorizer.get_feature_names_out()

print("\n=== Túi từ ===")
print(f"Kích thước ma trận X: {X.shape}")
print(f"Số từ trong từ điển: {len(words)}")

for label, name in [(0, "thường"), (1, "rác")]:
    counts = X[df["la_rac"].to_numpy() == label].sum(axis=0).A1
    top = counts.argsort()[::-1][:10]
    print(f"\n10 từ xuất hiện nhiều nhất ở lớp {name}:")
    for index in top:
        print(f"  {words[index]:<16} {int(counts[index])}")


=== Thông tin dữ liệu ===
Kích thước: (300, 2)
Số lượng từng lớp:
la_rac
0    180
1    120
Tỷ lệ thư rác: 0.4000

=== Túi từ ===
Kích thước ma trận X: (300, 40)
Số từ trong từ điển: 40

10 từ xuất hiện nhiều nhất ở lớp thường:
  thầy             74
  bài              73
  cáo              72
  chiều            68
  ạ                65
  tài              64
  tập              63
  gửi              61
  lớp              60
  nhóm             58

10 từ xuất hiện nhiều nhất ở lớp rác:
  tay              48
  ngay             47
  nhanh            44
  chót             43
  lúc              42
  giảm             42
  quà              41
  phí              40
  click            40
  miễn             39


## Bước 3 - Lọc tin nhắn rác


In [3]:
"""Bước 3: huấn luyện bộ lọc tin nhắn rác bằng MultinomialNB."""

from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "tin_nhan.csv")

vectorizer = CountVectorizer(token_pattern=r"(?u)\S+")
X = vectorizer.fit_transform(df["noi_dung"])
y = df["la_rac"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
model = MultinomialNB(alpha=1.0)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("=== Kết quả trên tập kiểm tra ===")
print(f"Tập huấn luyện: {X_train.shape[0]} mẫu")
print(f"Tập kiểm tra:   {X_test.shape[0]} mẫu")
print("Ma trận nhầm lẫn:")
print(confusion_matrix(y_test, y_pred))
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred):.4f}")

messages = [
    "trung thuong khuyen mai nhanh tay click ngay",
    "hop nhom bao cao chieu mai phong 305",
    "ban gui giup em tai lieu bai tap nhe",
]
probabilities = model.predict_proba(vectorizer.transform(messages))[:, 1]
print("\n=== Dự đoán tin nhắn mới ===")
for message, probability in zip(messages, probabilities):
    label = "rác" if probability >= 0.5 else "thường"
    print(f"{message!r}: P(rác) = {probability:.4f} -> {label}")


=== Kết quả trên tập kiểm tra ===
Tập huấn luyện: 225 mẫu
Tập kiểm tra:   75 mẫu
Ma trận nhầm lẫn:
[[45  0]
 [ 4 26]]
Accuracy : 0.9467
Precision: 1.0000
Recall   : 0.8667

=== Dự đoán tin nhắn mới ===
'trung thuong khuyen mai nhanh tay click ngay': P(rác) = 0.9992 -> rác
'hop nhom bao cao chieu mai phong 305': P(rác) = 0.0983 -> thường
'ban gui giup em tai lieu bai tap nhe': P(rác) = 0.0719 -> thường


## Bước 4 - Làm trơn Laplace


In [4]:
"""Bước 4: minh họa làm trơn Laplace cho Naive Bayes."""

from collections import Counter
from math import prod


corpus = [
    ("trung thuong ngay", 1),
    ("khuyen mai giam gia", 1),
    ("trung thuong click", 1),
    ("hop nhom chieu mai", 0),
    ("bai tap nhom nop mai", 0),
    ("thay gui tai lieu", 0),
]

vocabulary = sorted({word for text, _ in corpus for word in text.split()})
test_words = "trung thuong bai tap".split()
print(f"Số từ trong từ điển: {len(vocabulary)}")
print(f"Tin nhắn cần phân loại: {' '.join(test_words)!r}")


def class_score(label, alpha):
    documents = [text for text, target in corpus if target == label]
    counts = Counter(word for text in documents for word in text.split())
    total_words = sum(counts.values())
    prior = len(documents) / len(corpus)
    probabilities = [
        (counts[word] + alpha) / (total_words + alpha * len(vocabulary))
        for word in test_words
    ]
    return prior * prod(probabilities)


for alpha in [0, 1]:
    spam_score = class_score(1, alpha)
    normal_score = class_score(0, alpha)
    prediction = "rác" if spam_score > normal_score else "thường"
    print(f"\nalpha = {alpha}")
    print(f"Điểm lớp rác   : {spam_score:.10f}")
    print(f"Điểm lớp thường: {normal_score:.10f}")
    print(f"Dự đoán: {prediction}")

print("\nKhi alpha = 0, một từ chưa từng xuất hiện làm cả tích xác suất bằng 0.")
print("Làm trơn Laplace (alpha = 1) giúp mọi từ có xác suất khác 0.")


Số từ trong từ điển: 18
Tin nhắn cần phân loại: 'trung thuong bai tap'

alpha = 0
Điểm lớp rác   : 0.0000000000
Điểm lớp thường: 0.0000000000
Dự đoán: thường

alpha = 1
Điểm lớp rác   : 0.0000073212
Điểm lớp thường: 0.0000021656
Dự đoán: rác

Khi alpha = 0, một từ chưa từng xuất hiện làm cả tích xác suất bằng 0.
Làm trơn Laplace (alpha = 1) giúp mọi từ có xác suất khác 0.


## Bước 5 - Gaussian Naive Bayes


In [5]:
"""Bước 5: Gaussian Naive Bayes cho dữ liệu sinh viên liên tục."""

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "sinh_vien.csv")
X = df[["gio_on", "diem_giua_ky"]]
y = df["qua_mon"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

gaussian = GaussianNB()
logistic = LogisticRegression(max_iter=1000)
gaussian.fit(X_train, y_train)
logistic.fit(X_train, y_train)

print("=== So sánh độ chính xác ===")
print(f"GaussianNB         : {accuracy_score(y_test, gaussian.predict(X_test)):.4f}")
print(f"LogisticRegression : {accuracy_score(y_test, logistic.predict(X_test)):.4f}")

print("\n=== Tham số GaussianNB học được ===")
for index, label in enumerate(gaussian.classes_):
    means = gaussian.theta_[index]
    stds = np.sqrt(gaussian.var_[index])
    print(
        f"Lớp {label}: mean(gio_on, diem_giua_ky) = ({means[0]:.2f}, {means[1]:.2f}), "
        f"std = ({stds[0]:.2f}, {stds[1]:.2f}), prior = {gaussian.class_prior_[index]:.4f}"
    )

new_students = pd.DataFrame(
    [[5, 4], [13, 6], [20, 7], [28, 9]], columns=["gio_on", "diem_giua_ky"]
)
pass_probabilities = gaussian.predict_proba(new_students)[:, 1]
print("\n=== Dự đoán sinh viên mới ===")
for row, probability in zip(new_students.itertuples(index=False), pass_probabilities):
    print(f"({row.gio_on:>2}, {row.diem_giua_ky}) -> P(qua môn) = {probability:.4f}")


=== So sánh độ chính xác ===
GaussianNB         : 0.9000
LogisticRegression : 0.9000

=== Tham số GaussianNB học được ===
Lớp 0: mean(gio_on, diem_giua_ky) = (8.18, 3.99), std = (5.29, 1.92), prior = 0.4111
Lớp 1: mean(gio_on, diem_giua_ky) = (20.03, 6.70), std = (5.81, 1.50), prior = 0.5889

=== Dự đoán sinh viên mới ===
( 5, 4) -> P(qua môn) = 0.0137
(13, 6) -> P(qua môn) = 0.6550
(20, 7) -> P(qua môn) = 0.9855
(28, 9) -> P(qua môn) = 0.9999
